# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,LNG,49,1,0.998300,2026-05-14 11:08:46+00:00
1,NVDA,11,7,0.638814,2026-05-19 21:06:13+00:00
2,RKLB,10,5,0.306240,2026-05-19 22:35:31+00:00
3,NBIS,6,6,0.511017,2026-05-15 23:10:06+00:00
4,POET,15,3,0.785667,2026-05-18 20:02:54+00:00
5,DRAM,8,4,0.502900,2026-05-19 14:57:20+00:00
6,INTC,6,4,-0.023175,2026-05-18 19:46:52+00:00
7,OMER,17,1,0.995100,2026-05-18 16:05:11+00:00
8,AMD,5,4,-0.135325,2026-05-18 02:35:27+00:00
9,SPY,5,4,-0.513175,2026-05-18 19:22:09+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['LNG', 'NVDA', 'RKLB', 'NBIS']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
    analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": float(features_df["Close"].dropna().iloc[-1]),
        "latest_rsi": float(features_df["RSI"].dropna().iloc[-1]),
        "latest_macd": float(features_df["MACD"].dropna().iloc[-1]),
        "return_21d": float(features_df["21D Return"].dropna().iloc[-1]),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,LNG,251,246.770004,42.910796,-5.562246,-0.022112,4.076247,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
1,NBIS,251,197.729996,60.569329,18.410107,0.242335,7.552824,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
2,NVDA,251,220.610001,59.949333,8.604643,0.091804,3.083503,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
3,RKLB,251,127.309998,69.569904,14.136921,0.423094,4.594758,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== LNG latest metrics =====
                 Close   Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI      MACD  5D Return  21D Return
Date                                                                                                                                    
2026-05-13  239.380005  2120800           12.581166          262.075334           247.914369  35.216300 -5.348873  -0.084309   -0.078173
2026-05-14  241.080002  2055700           12.867159          260.916667           247.757763  36.926480 -5.860183  -0.023097   -0.061032
2026-05-15  241.839996  2682100           12.735967          259.606000           247.622171  37.718043 -6.133373   0.007205   -0.079546
2026-05-18  247.740005  2461000           12.039141          258.395668           247.624871  43.632243 -5.806858   0.029248   -0.013263
2026-05-19  246.770004  2223887           11.175847          257.145668           247.605286  42.910796 -5.562246   0.010069   -0.022112

===== NV

## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== LNG regression results =====
                 Close  Predicted Close
Date                                   
2026-05-06  261.420013       263.068680
2026-05-07  246.779999       254.903952
2026-05-08  240.110001       248.987076
2026-05-11  240.699997       245.123814
2026-05-12  244.309998       245.435998
2026-05-13  239.380005       242.732231
2026-05-14  241.080002       245.317328
2026-05-15  241.839996       245.732929
2026-05-18  247.740005       249.029326
2026-05-19  246.770004       246.889150

===== NVDA regression results =====
                 Close  Predicted Close
Date                                   
2026-05-06  207.830002       202.623473
2026-05-07  211.500000       206.389683
2026-05-08  215.199997       210.066476
2026-05-11  219.440002       213.950431
2026-05-12  220.779999       216.983417
2026-05-13  225.830002       219.890273
2026-05-14  235.740005       226.226550
2026-05-15  225.320007       219.216605
2026-05-18  222.320007       217.663335
2026-05

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "LNG",
    "NVDA",
    "RKLB",
    "NBIS"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/financial_metrics_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
